# Week 8 Capstone — Meeting-to-Action Assistant

Reuses Week 3 structured output, Week 4 validation, Week 5 MiniLM+FAISS RAG, Week 6 cross-encoder reranking, and Week 7 dynamic few-shot selection. A/B/C: A transcript only; B adds retrieved/reranked guidance; C also adds dynamic few-shot examples.

## Colab setup — install dependencies

**Run the next code cell once in a fresh Google Colab runtime, then restart the runtime before continuing.**

This notebook uses Colab's bundled PyTorch; do not reinstall `torch`. The retrieval stack is pinned to a known-compatible pair: `sentence-transformers==5.1.0` and `transformers==4.57.1`. This avoids the mixed Transformers 4/5 state that causes `find_pruneable_heads_and_indices` import failures.

After the install cell completes, choose **Runtime → Restart session**, then continue from the imports cell below. If this runtime previously installed a different `torch`, `transformers`, or `sentence-transformers`, use **Runtime → Disconnect and delete runtime** first, reconnect, run the install cell, restart once, and continue.

In [5]:
# %pip install -q "sentence-transformers==5.1.0" "transformers==4.57.1" faiss-cpu openai pandas numpy tabulate tqdm

In [1]:
import os,re,json,time,random,hashlib,numpy as np,pandas as pd,faiss
import torch, transformers, sentence_transformers
from sentence_transformers import SentenceTransformer,CrossEncoder
from IPython.display import display,Markdown
from tqdm.auto import tqdm
print("PyTorch:",torch.__version__,"| Transformers:",transformers.__version__,"| Sentence Transformers:",sentence_transformers.__version__)
try:
 from google.colab import userdata
 if not os.getenv("GEMINI_API_KEY"):os.environ["GEMINI_API_KEY"]=userdata.get("GEMINI_API_KEY")
except:pass
GM=os.getenv("GEMINI_MODEL","gemini-3.5-flash-lite");JM=os.getenv("GEMINI_JUDGE_MODEL",GM);QM="Qwen/Qwen2.5-0.5B-Instruct";CACHE={"s":{},"j":{}}
def K(x):return hashlib.sha256(json.dumps(x,sort_keys=True).encode()).hexdigest()
def A(m,p):
 from openai import OpenAI
 c=OpenAI(api_key=os.environ["GEMINI_API_KEY"],base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
 for i,w in enumerate([2,4,8,16,32,32]):
  try:return c.chat.completions.create(model=m,messages=[{"role":"user","content":p}],temperature=0)
  except Exception as e:
   if getattr(e,"status_code",None) not in {429,500,502,503,504} or i==5:raise
   d=w+random.random();tqdm.write(f"Gemini {getattr(e,'status_code',None)}; retry in {d:.1f}s");time.sleep(d)
def J(s):return json.loads(re.sub(r"^```(?:json)?\s*|\s*```$","",s.strip(),flags=re.I))
print("Gemini:",GM,"| Judge:",JM,"| Qwen:",QM)

PyTorch: 2.11.0+cpu | Transformers: 4.57.1 | Sentence Transformers: 5.1.0
Gemini: gemini-3.5-flash-lite | Judge: gemini-3.5-flash-lite | Qwen: Qwen/Qwen2.5-0.5B-Instruct


## Held-out set and reused retrieval components

In [2]:
C=[("E01","Maybe move Monday. No, keep Friday. I'll send QA.",["keep friday"],[("send qa","B")],[],[]),("E02","Export missing IDs. I'll ask Finance.",[],[("ask finance","B")],["missing ids"],[]),("E03","Could we use vendor B? Legal has not reviewed it. Do not choose yet.",["do not choose"],[],["legal has not reviewed"],[]),("E04","Weekly dashboard might help. No cadence decision. I'll collect examples.",[],[("collect examples","B")],[],[]),("E05","Import failed on invalid dates. I'll fix dates. Keep current mapping.",["keep current mapping"],[("fix dates","B")],["invalid dates"],[]),("E06","Use Sep 1-Sep 30 for both reports. I'll rerun them.",["use sep 1 sep 30"],[("rerun","B")],[],[]),("E07","We could automate email. Not in v1; human review required. I'll document the gate.",["human review required"],[("document gate","A")],[],[]),("E08","Duplicate customers cause bad totals. I'll test customer ID.",[],[("test customer id","B")],["duplicate customers"],[]),("E09","Should we remove notes? Not yet. Keep shorter form. I'll measure time.",["keep shorter form"],[("measure time","A")],[],["remove notes"]),("E10","SSO effort unknown. Do not promise it. I'll ask Engineering. Need separate coreference model?",["do not promise"],[("ask engineering","B")],["sso effort unknown"],["coreference model"])]
R=["Decisions require explicit commitment.","Maybe/could/might suggestions are not decisions unless later committed.","Actions are future work; never invent an owner.","Blockers prevent, delay, or risk progress.","Questions must remain unresolved.","Extracted items need transcript evidence."]
S=[("Maybe delay. No, keep Friday.","decision"),("Could we use vendor B? Compare terms first.","no decision"),("Missing IDs. I'll ask Finance.","action"),("Legal not reviewed.","blocker"),("Should we remove notes?","question")]
display(pd.DataFrame([{"case":x[0],"transcript":x[1]} for x in C]));display(pd.DataFrame({"RAG guidance":R}));display(pd.DataFrame(S,columns=["few-shot input","target"]))
b=tqdm(total=4,desc="Loading retrieval components");E=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2");b.update(1);X=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2");b.update(1)
def I(d):v=E.encode(d,normalize_embeddings=True,show_progress_bar=False).astype("float32");z=faiss.IndexFlatIP(v.shape[1]);z.add(v);return z
RI=I(R);b.update(1);SI=I([x[0] for x in S]);b.update(1);b.close()
def ret(q,d,z,t=lambda x:x,k=3):v=E.encode([q],normalize_embeddings=True,show_progress_bar=False).astype("float32");_,ids=z.search(v,min(6,len(d)));c=[d[int(i)] for i in ids[0] if i>=0];s=X.predict([(q,t(x)) for x in c],show_progress_bar=False);return [x for _,x in sorted(zip(s,c),reverse=True)[:k]]
display(pd.DataFrame({"sample retrieved rules":ret(C[0][1],R,RI,k=4)}))

,case,transcript
0,E01,"Maybe move Monday. No, keep Friday. I'll send QA."
1,E02,Export missing IDs. I'll ask Finance.
2,E03,Could we use vendor B? Legal has not reviewed ...
3,E04,Weekly dashboard might help. No cadence decisi...
4,E05,Import failed on invalid dates. I'll fix dates...
5,E06,Use Sep 1-Sep 30 for both reports. I'll rerun ...
6,E07,We could automate email. Not in v1; human revi...
7,E08,Duplicate customers cause bad totals. I'll tes...
8,E09,Should we remove notes? Not yet. Keep shorter ...
9,E10,SSO effort unknown. Do not promise it. I'll as...


,RAG guidance
0,Decisions require explicit commitment.
1,Maybe/could/might suggestions are not decision...
2,Actions are future work; never invent an owner.
3,"Blockers prevent, delay, or risk progress."
4,Questions must remain unresolved.
5,Extracted items need transcript evidence.


,few-shot input,target
0,"Maybe delay. No, keep Friday.",decision
1,Could we use vendor B? Compare terms first.,no decision
2,Missing IDs. I'll ask Finance.,action
3,Legal not reviewed.,blocker
4,Should we remove notes?,question


Loading retrieval components:   0%|          | 0/4 [00:00<?, ?it/s]

,sample retrieved rules
0,Maybe/could/might suggestions are not decision...
1,Questions must remain unresolved.
2,Decisions require explicit commitment.
3,"Blockers prevent, delay, or risk progress."


## A/B/C evaluation, judge, failure, and mitigation

Progress bars expose the 30 synthesis cases, 10 judge cases, and mitigation pass.

In [3]:
P='''Return JSON only: {"executive_summary":"string","decisions":[{"decision":"string","evidence":"string"}],"action_items":[{"description":"string","owner":"string","evidence":"string"}],"blockers":[{"blocker":"string","evidence":"string"}],"follow_up_questions":[{"question":"string","evidence":"string"}]}. Only confirmed decisions; do not invent owners; blockers must delay/risk progress; unresolved questions only; include evidence.''';REQ=["executive_summary","decisions","action_items","blockers","follow_up_questions"]
def ex(t,g):
 k=K([GM,g,t])
 if k in CACHE["s"]:return CACHE["s"][k]
 p=[P];L={"rules":[],"examples":[]}
 if g in "BC":L["rules"]=ret(t,R,RI,k=4);p+=["Guidance: "+str(L["rules"])]
 if g=="C":L["examples"]=ret(t,S,SI,lambda x:x[0],3);p+=["Examples: "+str(L["examples"])]
 st=time.time()
 try:o=J(A(GM,chr(10).join(p+["Transcript:",t])).choices[0].message.content)
 except:o={}
 r={"brief":o,"valid":isinstance(o,dict) and all(x in o for x in REQ),"lat":time.time()-st,"log":L};CACHE["s"][k]=r;return r
def T(x):return set(re.findall(r"[a-z0-9]+",str(x).lower()))
def f(p,g):
 u=set();n=0
 for x in p:
  z=[(len(T(x)&T(y))/max(1,len(T(x)|T(y))),i) for i,y in enumerate(g) if i not in u]
  if z and max(z)[0]>=.25:n+=1;u.add(max(z)[1])
 a=n/len(p) if p else (1 if not g else 0);b=n/len(g) if g else (1 if not p else 0);return 2*a*b/(a+b) if a+b else 0
def sc(c,r):
 b=r["brief"] if r["valid"] else {};d=[x.get("decision","") for x in b.get("decisions",[])];a=[x.get("description","") for x in b.get("action_items",[])];o=[str(x.get("owner") or "") for x in b.get("action_items",[])];bl=[x.get("blocker","") for x in b.get("blockers",[])];q=[x.get("question","") for x in b.get("follow_up_questions",[])];ow=1 if not c[3] else int(any(i<len(o) and o[i].strip().lower()==str(go or "").strip().lower() for i in range(len(o)) for _,go in c[3]));return {"case":c[0],"decision_f1":f(d,c[2]),"action_f1":f(a,[x[0] for x in c[3]]),"owner_accuracy":ow,"blocker_f1":f(bl,c[4]),"question_f1":f(q,c[5]),"valid":int(r["valid"]),"latency_s":r["lat"]}
RES={};rows=[]
if os.getenv("GEMINI_API_KEY"):
 with tqdm(total=30,desc="A/B/C synthesis evaluation") as bar:
  for g in "ABC":
   RES[g]={};bar.set_postfix(config=g)
   for c in C:r=ex(c[1],g);RES[g][c[0]]=r;x=sc(c,r);x["config"]=g;rows.append(x);bar.update(1)
 M=pd.DataFrame(rows);SUM=M.groupby("config").mean(numeric_only=True).round(3);SUM["score"]=SUM[["decision_f1","action_f1","owner_accuracy","blocker_f1","question_f1","valid"]].mean(axis=1).round(3);BEST=SUM.score.idxmax();display(M);display(SUM);print("Selected",BEST)
else:print("API-BLOCKED")
JD=pd.DataFrame();JS=pd.DataFrame()
if RES:
 z=[]
 for c in tqdm(C,desc="LLM-as-judge"):
  k=K([JM,c[0],RES[BEST][c[0]]["brief"]]);p={"transcript":c[1],"candidate":RES[BEST][c[0]]["brief"],"gold":c[2:]}
  try:j=CACHE["j"].get(k) or J(A(JM,'Score 1-5 on correctness, grounding, operational_usefulness. Return JSON only with those keys plus "reason".'+chr(10)+json.dumps(p)).choices[0].message.content);CACHE["j"][k]=j;z.append({"case":c[0],**j})
  except Exception as e:tqdm.write(f"judge {c[0]}: {e}")
 JD=pd.DataFrame(z)
 if not JD.empty:JS=JD[["correctness","grounding","operational_usefulness"]].mean().to_frame("average").round(3);display(JD);display(JS)
FAIL=None;BEF=AFT=None
if RES:
 s={g:{c[0]:sc(c,RES[g][c[0]]) for c in C} for g in "ABC"};ch=co=None
 for c in C:
  if s["C"][c[0]]["decision_f1"]<s["B"][c[0]]["decision_f1"]:ch,co=c[0],"dynamic few-shot";break
 if ch is None:
  for c in C:
   if s["B"][c[0]]["decision_f1"]<s["A"][c[0]]["decision_f1"]:ch,co=c[0],"retrieval/reranking";break
 if ch is None:ch=min((c[0] for c in C),key=lambda x:s[BEST][x]["decision_f1"]);co="generation"
 FAIL={"case":ch,"component":co}
 def mit(b):
  z=json.loads(json.dumps(b));H=["maybe","could","might"];Q=["agreed","keep","use ","do not","required","will"];z["decisions"]=[d for d in z.get("decisions",[]) if not(any(x in str(d.get("evidence") or "").lower() for x in H) and not any(x in str(d.get("evidence") or "").lower() for x in Q))];return z
 x=[];y=[]
 for c in tqdm(C,desc="Mitigation check"):r=RES[BEST][c[0]];x.append(sc(c,r)["decision_f1"]);r2=dict(r);r2["brief"]=mit(r["brief"]) if r["valid"] else {};r2["valid"]=all(k in r2["brief"] for k in REQ);y.append(sc(c,r2)["decision_f1"])
 BEF,AFT=round(float(np.mean(x)),3),round(float(np.mean(y)),3);print(FAIL,"Mitigation",BEF,"->",AFT)

A/B/C synthesis evaluation:   0%|          | 0/30 [00:00<?, ?it/s]

Gemini 429; retry in 2.1s
Gemini 429; retry in 2.9s
Gemini 429; retry in 2.4s
Gemini 429; retry in 4.8s
Gemini 429; retry in 8.7s
Gemini 429; retry in 16.5s


,case,decision_f1,action_f1,owner_accuracy,blocker_f1,question_f1,valid,latency_s,config
0,E01,0.000000,1.000000,0,1.0,1.000000,1,12.198707,A
1,E02,1.000000,1.000000,0,1.0,1.000000,1,1.399582,A
2,E03,0.000000,1.000000,1,0.0,1.000000,0,1.017438,A
3,E04,1.000000,1.000000,0,1.0,0.000000,1,1.054309,A
4,E05,1.000000,1.000000,0,1.0,1.000000,1,1.008130,A
5,E06,1.000000,0.000000,0,1.0,1.000000,1,1.044777,A
6,E07,1.000000,1.000000,0,1.0,1.000000,1,1.034453,A
7,E08,0.000000,0.000000,0,0.0,1.000000,1,0.942686,A
8,E09,1.000000,1.000000,0,1.0,0.000000,1,0.984126,A
9,E10,0.000000,0.000000,0,1.0,0.666667,1,0.904350,A


,decision_f1,action_f1,owner_accuracy,blocker_f1,question_f1,valid,latency_s,score
config,,,,,,,,
A,0.600,0.700,0.1,0.8,0.767,0.9,2.159,0.644
B,0.667,0.867,0.1,0.7,0.667,1.0,6.031,0.667
C,0.700,1.000,0.1,0.6,0.700,1.0,0.826,0.683


Selected C


...

Gemini 429; retry in 2.4s
Gemini 429; retry in 2.3s
Gemini 429; retry in 2.6s
Gemini 429; retry in 4.0s
Gemini 429; retry in 8.6s


,case,correctness,grounding,operational_usefulness,reason
0,E01,4,5,4,The executive summary accurately captures the ...
1,E02,3,4,4,The candidate captures the core intent well (a...
2,E03,5,5,5,The candidate accurately captures the essence ...
3,E04,4,5,4,The summary accurately reflects the transcript...
4,E05,5,5,5,The candidate accurately extracts the decision...
5,E06,5,5,5,The candidate accurately captures the decision...
6,E07,5,5,5,The candidate accurately captures the decision...
7,E08,4,5,4,The candidate accurately identifies the core p...
8,E09,5,5,5,The candidate accurately extracts the decision...
9,E10,4,5,5,The candidate accurately captures the key poin...


,average
correctness,4.4
grounding,4.9
operational_usefulness,4.6


Mitigation check:   0%|          | 0/10 [00:00<?, ?it/s]

{'case': 'E09', 'component': 'dynamic few-shot'} Mitigation 0.7 -> 0.7


## Gemini vs. small local Qwen

Gemini batch classification and local Qwen inference both expose progress.

In [4]:
D=[("Maybe move launch.","context"),("Keep Friday.","decision"),("I'll send QA.","action_item"),("SSO unknown.","blocker"),("What is SSO effort?","follow_up_question"),("No cadence decision.","context"),("Keep mapping.","decision"),("I'll fix dates.","action_item"),("Legal not reviewed.","blocker"),("Which agreement can Legal clear?","follow_up_question")];GDF=QDF=CMP=pd.DataFrame()
if os.getenv("GEMINI_API_KEY"):
 try:
  with tqdm(total=1,desc="Gemini classification batch") as b:p='Classify each span as decision, action_item, blocker, follow_up_question, context. Return JSON only as {"labels":[]}.'+chr(10)+chr(10).join(f"{i+1}. {t}" for i,(t,_) in enumerate(D));st=time.time();o=J(A(GM,p).choices[0].message.content);lat=(time.time()-st)/len(D);b.update(1)
  GDF=pd.DataFrame([{"gold":g,"pred":q,"correct":g==q,"latency":lat} for (_,g),q in zip(D,o["labels"])]);display(GDF)
 except Exception as e:print(e)
try:
 import torch
 from transformers import AutoTokenizer,AutoModelForCausalLM
 print("Loading local Qwen tokenizer/model...");tok=AutoTokenizer.from_pretrained(QM);m=AutoModelForCausalLM.from_pretrained(QM,torch_dtype="auto");m.eval()
 def qc(t):
  x=tok(tok.apply_chat_template([{"role":"user","content":"Return one label only: decision, action_item, blocker, follow_up_question, context. "+t}],tokenize=False,add_generation_prompt=True),return_tensors="pt");st=time.time()
  with torch.no_grad():y=m.generate(**x,max_new_tokens=8,do_sample=False)
  return re.sub(r"[^a-z_]","",tok.decode(y[0][x["input_ids"].shape[1]:],skip_special_tokens=True).strip().lower()),time.time()-st
 z=[]
 for t,g in tqdm(D,desc="Qwen local classification"):q,l=qc(t);z.append({"gold":g,"pred":q,"correct":g==q,"latency":l})
 QDF=pd.DataFrame(z);display(QDF)
except Exception as e:print(e)
z=[]
if not GDF.empty:z.append({"model":"Gemini","accuracy":GDF.correct.mean(),"latency":GDF.latency.mean()})
if not QDF.empty:z.append({"model":"Qwen2.5-0.5B","accuracy":QDF.correct.mean(),"latency":QDF.latency.mean()})
CMP=pd.DataFrame(z)
if not CMP.empty:display(CMP.round(3))

Gemini classification batch:   0%|          | 0/1 [00:00<?, ?it/s]

,gold,pred,correct,latency
0,context,"{'span': 'Maybe move launch.', 'label': 'decis...",False,0.103859
1,decision,"{'span': 'Keep Friday.', 'label': 'decision'}",False,0.103859
2,action_item,"{'span': 'I'll send QA.', 'label': 'action_item'}",False,0.103859
3,blocker,"{'span': 'SSO unknown.', 'label': 'blocker'}",False,0.103859
4,follow_up_question,"{'span': 'What is SSO effort?', 'label': 'foll...",False,0.103859
5,context,"{'span': 'No cadence decision.', 'label': 'blo...",False,0.103859
6,decision,"{'span': 'Keep mapping.', 'label': 'decision'}",False,0.103859
7,action_item,"{'span': 'I'll fix dates.', 'label': 'action_i...",False,0.103859
8,blocker,"{'span': 'Legal not reviewed.', 'label': 'bloc...",False,0.103859
9,follow_up_question,"{'span': 'Which agreement can Legal clear?', '...",False,0.103859


Loading local Qwen tokenizer/model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen local classification:   0%|          | 0/10 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,gold,pred,correct,latency
0,context,decisionaction_itemblockerfollow,False,4.822062
1,decision,decision,True,3.762520
2,action_item,imreadytoassistwithyourquestion,False,4.901952
3,blocker,decision,False,3.807128
4,follow_up_question,thetermssoefforttypically,False,5.720746
5,context,decision,False,4.060481
6,decision,decisionaction_itemblocker,False,4.537988
7,action_item,imsorrybutineedmore,False,4.929944
8,blocker,decision,False,3.795704
9,follow_up_question,thedecisionisthefinalstepinresolving,False,4.861576


,model,accuracy,latency
0,Gemini,0.0,0.104
1,Qwen2.5-0.5B,0.1,4.520


# Research Note — Engineering Basis

- **RAG:** Lewis et al. (2020), https://arxiv.org/abs/2005.11401 — retrieval conditions generation on retrieved context; A→B tests value here.
- **Dense retrieval:** Reimers & Gurevych (2019), https://arxiv.org/abs/1908.10084 — Sentence-BERT supports efficient semantic retrieval.
- **Reranking:** Nogueira & Cho (2019), https://arxiv.org/abs/1901.04085 — BERT reranking supports a dense-retrieval-then-cross-encoder design.
- **LLM-as-judge:** Zheng et al. (2023), https://arxiv.org/abs/2306.05685 — judge bias makes deterministic metrics primary and judge scores supplemental.
- **Local model:** Qwen2.5, https://arxiv.org/abs/2412.15115 — small open models justify testing a narrow local role rather than assuming full replacement.

**Research position:** retain retrieval/reranking and few-shot context only when ablation shows downstream value; use Qwen in the critical path only if measured accuracy is competitive.

# Automated Findings Memo — Measured Results/Conclusions

In [5]:
def memo():
 if not RES or JD.empty or FAIL is None:return None
 j=chr(10).join(f"- **{i}:** {float(r['average']):.3f}" for i,r in JS.iterrows());q="comparison incomplete"
 if len(CMP)>=2:q=f"Qwen accuracy {CMP.iloc[1].accuracy:.3f} vs Gemini {CMP.iloc[0].accuracy:.3f}; latency {CMP.iloc[1].latency:.3f}s vs {CMP.iloc[0].latency:.3f}s."
 s=f'''## A/B/C metrics
{SUM.to_markdown()}

**Selected:** {BEST}

## Judge averages
{j}

**Failure:** {FAIL["case"]} — {FAIL["component"]}

**Mitigation decision F1:** {BEF} → {AFT}

**Gemini vs Qwen:** {q}''';display(Markdown(s));return s
FINDINGS=memo()

## A/B/C metrics
| config   |   decision_f1 |   action_f1 |   owner_accuracy |   blocker_f1 |   question_f1 |   valid |   latency_s |   score |
|:---------|--------------:|------------:|-----------------:|-------------:|--------------:|--------:|------------:|--------:|
| A        |         0.6   |       0.7   |              0.1 |          0.8 |         0.767 |     0.9 |       2.159 |   0.644 |
| B        |         0.667 |       0.867 |              0.1 |          0.7 |         0.667 |     1   |       6.031 |   0.667 |
| C        |         0.7   |       1     |              0.1 |          0.6 |         0.7   |     1   |       0.826 |   0.683 |

**Selected:** C

## Judge averages
- **correctness:** 4.400
- **grounding:** 4.900
- **operational_usefulness:** 4.600

**Failure:** E09 — dynamic few-shot

**Mitigation decision F1:** 0.7 → 0.7

**Gemini vs Qwen:** Qwen accuracy 0.100 vs Gemini 0.000; latency 4.520s vs 0.104s.

# Final Defended Conclusion

In [6]:
def conclusion():
 if FINDINGS is None:return None
 a,b,c=[float(SUM.loc[x,"score"]) for x in "ABC"];q="Qwen comparison incomplete; exclude it from the critical path."
 if len(CMP)>=2:
  g,w=float(CMP.iloc[0].accuracy),float(CMP.iloc[1].accuracy);q=(f"Qwen accuracy {w:.3f} is within 0.05 of Gemini {g:.3f}; use it only as an optional local classifier." if w>=g-.05 else f"Qwen accuracy {w:.3f} trails Gemini {g:.3f}; do not put it in the critical path.")
 s=f'''## Defended architecture decision
Use **configuration {BEST}** because it achieved the highest held-out extraction score. The ablation decides whether researched techniques earn a place here: **A→B {a:.3f}→{b:.3f}; B→C {b:.3f}→{c:.3f}.**

Treat the 10-case LLM-as-judge pass as supporting evidence, not selection authority. The required failure was **{FAIL["case"]}**, traced to **{FAIL["component"]}**. Mitigation changed decision F1 **{BEF:.3f}→{AFT:.3f}**. For local inference, {q}

### Conclusion
**Ship the simplest winning A/B/C configuration; keep deterministic validation and metrics around Gemini; retain RAG/reranking or dynamic few-shot context only when measured ablations justify them; treat LLM-as-judge as supplemental; use Qwen only for a narrow role when its measured accuracy supports the tradeoff.**''';display(Markdown(s));return s
DEFENDED_CONCLUSION=conclusion()
p=[]
if not RES:p+=["A/B/C"]
if JD.empty or len(JD)!=10:p+=["judge"]
if FAIL is None:p+=["failure"]
if CMP.empty or len(CMP)<2:p+=["Qwen"]
if FINDINGS is None:p+=["findings"]
if DEFENDED_CONCLUSION is None:p+=["conclusion"]
print("READY FOR SUBMISSION" if not p else "NOT READY: "+", ".join(p))

## Defended architecture decision
Use **configuration C** because it achieved the highest held-out extraction score. The ablation decides whether researched techniques earn a place here: **A→B 0.644→0.667; B→C 0.667→0.683.**

Treat the 10-case LLM-as-judge pass as supporting evidence, not selection authority. The required failure was **E09**, traced to **dynamic few-shot**. Mitigation changed decision F1 **0.700→0.700**. For local inference, Qwen accuracy 0.100 is within 0.05 of Gemini 0.000; use it only as an optional local classifier.

### Conclusion
**Ship the simplest winning A/B/C configuration; keep deterministic validation and metrics around Gemini; retain RAG/reranking or dynamic few-shot context only when measured ablations justify them; treat LLM-as-judge as supplemental; use Qwen only for a narrow role when its measured accuracy supports the tradeoff.**

READY FOR SUBMISSION
